# Federated AMR — Gentamicin

**Federated learning across 4 hospital sites** (DRIAMS A/B/C/D) using Flower.

| Component | Detail |
|---|---|
| Drug | **Gentamicin** |
| Species | All species, species-stratified splits |
| Architecture | MLP: 6000 → 512 → 256 → 128 → 2 |
| Strategies | FedAvg MLP, FedProx (μ=0.01/0.1/0.5), FedAvg LR, FedRF tree collection |
| Checkpoints | Per-client + global model saved every round |
| Mixing | Small sites train multiple seeds, average weights (MLP only) |

Flower simulation (Ray backend). Compatible with Google Colab.

In [ ]:
!pip install "flwr[simulation]" maldideepkit maldiamrkit seaborn --quiet

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print('Google Drive mounted')
    IN_COLAB = True
except ImportError:
    print('Running locally')
    IN_COLAB = False

In [ ]:
import warnings, copy, os, io, math
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import (train_test_split, GridSearchCV,
                                     RandomizedSearchCV, cross_val_predict,
                                     StratifiedKFold)
from sklearn.metrics import (balanced_accuracy_score, roc_auc_score)

from maldideepkit.attention.mlp import SpectralAttentionMLP
from maldideepkit.base.data import fit_input_transform, apply_input_transform
from maldiamrkit.evaluation import stratified_species_drug_split

import flwr as fl
import joblib

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=UserWarning)
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

FL_DEVICE = "cpu"  # Ray workers don't have GPU access
print(f"Flower version: {fl.__version__}")
print(f"FL device: {FL_DEVICE}")

In [ ]:
# ── Paths ──
if IN_COLAB:
    DRYAD = Path("/content/drive/MyDrive/Flower/DRIAMS-DataSet")
else:
    DRYAD = Path("/media/asd/8f69beed-e984-445f-b8b3-abbb6a1a4b3f/Dryad-DataSet")

DRUG_NAME = "Gentamicin"
DRUG_CSV  = "Gentamicin"

OUT_DIR = Path("./results")
MODEL_DIR = Path("./models")
OUT_DIR.mkdir(exist_ok=True)
MODEL_DIR.mkdir(exist_ok=True)

SITES_PATHS = {
    "A": DRYAD / "Processed/Proc_DRIAMS-A" / DRUG_NAME / "data.csv",
    "B": DRYAD / "Processed/Proc_DRIAMS-B" / DRUG_NAME / "data.csv",
    "C": DRYAD / "Processed/Proc_DRIAMS-C" / DRUG_NAME / "data.csv",
    "D": DRYAD / "Processed/Proc_DRIAMS-D" / DRUG_NAME / "data.csv",
}
SITE_ORDER = ["A", "B", "C", "D"]

BEST_PARAMS_CSV = DRYAD / "Processed/Processing/Analysis/07-Dedicated-MLP-Aggregated/07-01-results_dedicated_lr_mlp/best_params.csv"

print(f"Drug: {DRUG_NAME}")
print(f"CSV lookup: {DRUG_CSV}")
print(f"Best params from: {BEST_PARAMS_CSV}")

In [ ]:
# ── Shared constants ──
THRESHOLDS = np.linspace(0.05, 0.95, 91)
NUM_ROUNDS = 30
NUM_RF_ROUNDS = 10
LOCAL_EPOCHS = 1
BATCH_SIZE = 16
FEDPROX_MUS = [0.01, 0.1, 0.5]

RF_PARAM_GRID = {
    "n_estimators": [100, 200, 300, 500],
    "max_depth": [10, 20, 30, None],
    "min_samples_leaf": [2, 5, 10],
    "class_weight": ["balanced", "balanced_subsample"],
}

def n_mixes(n_train):
    if n_train < 1500: return 3
    if n_train < 5000: return 2
    return 1

In [ ]:
# ── Load pre-computed LR + MLP params from 07-01 ──
params_df = pd.read_csv(BEST_PARAMS_CSV)
row = params_df[params_df["Drug"] == DRUG_CSV]
if len(row) == 0:
    print(f"WARNING: drug '{DRUG_CSV}' not found in best_params.csv. Using defaults.")
    BEST_LR_C = 2.5e-04; BEST_MLP_LR = 8.2e-05; BEST_MLP_DH = 0.7
else:
    r = row.iloc[0]
    BEST_LR_C = float(r["LR"].replace("C=", ""))
    mlp = r["MLP"]
    BEST_MLP_LR = float(mlp.split("lr=")[1].split(" ")[0])
    BEST_MLP_DH = float(mlp.split("drop=")[1])
print(f"  LR:   C={BEST_LR_C:.2e}")
print(f"  MLP:  lr={BEST_MLP_LR:.2e}  dropout={BEST_MLP_DH:.1f}")

In [ ]:
# ── Load drug CSV from all 4 sites ──
raw_data = {}
for site, path in SITES_PATHS.items():
    df = pd.read_csv(path)
    bin_cols = [c for c in df.columns if c.startswith("bin_")]
    X = df[bin_cols].to_numpy(dtype="float32")
    y = df["label"].to_numpy(dtype="int64")
    species = df["species"].values
    raw_data[site] = (X, y, species)
    n_r, n_s = (y == 1).sum(), (y == 0).sum()
    n_sp = len(np.unique(species))
    print(f"  Site {site}: {len(y)} samples ({n_s} S, {n_r} R, {n_r/len(y)*100:.1f}% R, {n_sp} species)")

total = sum(len(raw_data[s][1]) for s in SITE_ORDER)
print(f"\nTotal pooled: {total} samples")

In [ ]:
# ── Per-site species-stratified 90/10 train/test ──
client_train = {}
client_test = {}

for site in SITE_ORDER:
    X, y, sp = raw_data[site]
    n = len(y)
    idx = np.arange(n).reshape(-1, 1)
    idx_train, idx_val, _, _ = stratified_species_drug_split(
        idx, y, species=sp, test_size=0.10, random_state=SEED)
    idx_train = idx_train.flatten().astype(int)
    idx_val = idx_val.flatten().astype(int)
    client_train[site] = (X[idx_train], y[idx_train])
    client_test[site] = (X[idx_val], y[idx_val])
    print(f"  Site {site}: train={len(idx_train)} test={len(idx_val)}")

pooled_X_train = np.concatenate([client_train[s][0] for s in SITE_ORDER])
pooled_y_train = np.concatenate([client_train[s][1] for s in SITE_ORDER])
print(f"\nPooled train: {len(pooled_X_train)} (for cross-site baseline)")

In [ ]:
# ── Per-site preprocessing ──
client_train_pp = {}
client_test_pp = {}
for site in SITE_ORDER:
    X_tr, y_tr = client_train[site]
    X_te, y_te = client_test[site]
    state = fit_input_transform(X_tr, "log1p+standardize")
    client_train_pp[site] = (apply_input_transform(X_tr, state), y_tr)
    client_test_pp[site] = (apply_input_transform(X_te, state), y_te)
    print(f"  Site {site}: preprocessed")
print("Per-site preprocessing done.")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# OPTION A: RF per-site grid → worst-site optimization
# ═══════════════════════════════════════════════════════════════════════════

print("\n=== RF Per-Site Grid (Option A) ===")

MAX_GS_SAMPLES = 5000
site_rf_scores = {}
for site in SITE_ORDER:
    X_tr, y_tr = client_train_pp[site]
    if len(X_tr) > MAX_GS_SAMPLES:
        idx = np.random.RandomState(SEED).choice(len(X_tr), MAX_GS_SAMPLES, replace=False)
        X_gs, y_gs = X_tr[idx], y_tr[idx]
    else:
        X_gs, y_gs = X_tr, y_tr
    grid = RandomizedSearchCV(
        RandomForestClassifier(oob_score=True, random_state=SEED, n_jobs=-1),
        param_distributions=RF_PARAM_GRID, n_iter=20, cv=2,
        scoring="balanced_accuracy", n_jobs=-1, random_state=SEED)
    grid.fit(X_gs, y_gs)
    # Store per-combo scores
    for i, params in enumerate(grid.cv_results_["params"]):
        key = str(params)
        score = grid.cv_results_["mean_test_score"][i]
        site_rf_scores.setdefault(key, []).append(score)
    print(f"  Site {site}: {len(grid.cv_results_['params'])} combos tested")

# Worst-site optimization: pick combo with best minimum score across sites
best_worst = -1
best_rf_combo = None
for combo_str, scores in site_rf_scores.items():
    worst = min(scores)
    if worst > best_worst:
        best_worst = worst
        import ast
        best_rf_combo = ast.literal_eval(combo_str)
RF_PARAMS = best_rf_combo
print(f"\n  Best RF (worst-site BA={best_worst:.4f}): {RF_PARAMS}")
RF_TREES_PER_ROUND = RF_PARAMS["n_estimators"]

# RF threshold: per-site CV, worst-site optimal
rf_thresh_scores = {t: [] for t in THRESHOLDS}
for site in SITE_ORDER:
    X_tr, y_tr = client_train_pp[site]
    cv_proba = cross_val_predict(
        RandomForestClassifier(**RF_PARAMS, random_state=SEED, n_jobs=-1),
        X_tr, y_tr, cv=2, method="predict_proba", n_jobs=-1)[:, 1]
    for t in THRESHOLDS:
        rf_thresh_scores[t].append(balanced_accuracy_score(y_tr, cv_proba >= t))

BEST_RF_THRESH = max(THRESHOLDS, key=lambda t: min(rf_thresh_scores[t]))
print(f"  RF threshold (worst-site): {BEST_RF_THRESH:.3f}")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# Option A: Per-site threshold tuning → worst-site optimal
# ═══════════════════════════════════════════════════════════════════════════

def worst_site_threshold(model_class, model_kwargs, cv=3):
    """Tune threshold per-site, return threshold that maximizes worst-site BA."""
    scores = {t: [] for t in THRESHOLDS}
    for site in SITE_ORDER:
        X_tr, y_tr = client_train_pp[site]
        cv_proba = cross_val_predict(
            model_class(**model_kwargs, random_state=SEED),
            X_tr, y_tr, cv=cv, method="predict_proba", n_jobs=-1)[:, 1]
        for t in THRESHOLDS:
            scores[t].append(balanced_accuracy_score(y_tr, cv_proba >= t))
    best_t = max(THRESHOLDS, key=lambda t: min(scores[t]))
    return best_t

print("\n=== Threshold Tuning (Option A) ===")
BEST_LR_THRESH = worst_site_threshold(
    LogisticRegression,
    {"C": BEST_LR_C, "penalty": "l2", "solver": "lbfgs", "class_weight": "balanced", "max_iter": 5000})
print(f"  LR threshold: {BEST_LR_THRESH:.3f}")

# MLP threshold: use cross_val_predict-like approach per site
mlp_thresh_scores = {t: [] for t in THRESHOLDS}
for site in SITE_ORDER:
    X_tr, y_tr = client_train_pp[site]
    # Simple per-site CV for threshold: split in 3, train tiny MLPs
    kf = StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED)
    all_proba = np.zeros(len(y_tr))
    for ti, vi in kf.split(X_tr, y_tr):
        m = SpectralAttentionMLP(
            input_dim=6000, n_classes=2, hidden_dim=512, head_dims=(256, 128),
            dropout_high=BEST_MLP_DH, dropout_low=BEST_MLP_DH/2.0, use_attention=False)
        m.to(FL_DEVICE)
        ds = DatasetFromNumpy(X_tr[ti], y_tr[ti])
        dl = DataLoader(ds, batch_size=64, shuffle=True)
        opt = torch.optim.AdamW(m.parameters(), lr=BEST_MLP_LR, weight_decay=1e-3)
        crit = nn.CrossEntropyLoss()
        for _ in range(30):
            m.train()
            for xb, yb in dl:
                xb, yb = xb.to(FL_DEVICE), yb.to(FL_DEVICE)
                opt.zero_grad(); crit(m(xb), yb).backward(); opt.step()
        m.eval()
        X_v = torch.tensor(X_tr[vi], dtype=torch.float32).to(FL_DEVICE)
        with torch.no_grad():
            all_proba[vi] = F.softmax(m(X_v), dim=1).cpu().numpy()[:, 1]
    for t in THRESHOLDS:
        mlp_thresh_scores[t].append(balanced_accuracy_score(y_tr, all_proba >= t))

BEST_MLP_THRESH = max(THRESHOLDS, key=lambda t: min(mlp_thresh_scores[t]))
print(f"  MLP threshold: {BEST_MLP_THRESH:.3f}")

class DatasetFromNumpy(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)
    def __len__(self): return len(self.X)
    def __getitem__(self, i): return self.X[i], self.y[i]

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# CENTRALIZED MLP (pooled data, pre-computed params)
# ═══════════════════════════════════════════════════════════════════════════

print("\n=== Centralized MLP ===")
# Pooled preprocessing
state_pool = fit_input_transform(pooled_X_train, "log1p+standardize")
X_pool_pp = apply_input_transform(pooled_X_train, state_pool)

# Preprocess all test sets with pooled state
pooled_test_sets = {}
for site in SITE_ORDER:
    X_te, y_te = client_test[site]
    pooled_test_sets[site] = (apply_input_transform(X_te, state_pool), y_te)
combined_X_test = np.concatenate([pooled_test_sets[s][0] for s in SITE_ORDER])
combined_y_test = np.concatenate([pooled_test_sets[s][1] for s in SITE_ORDER])

# Train centralized MLP
cent_model = SpectralAttentionMLP(
    input_dim=6000, n_classes=2, hidden_dim=512, head_dims=(256, 128),
    dropout_high=BEST_MLP_DH, dropout_low=BEST_MLP_DH/2.0, use_attention=False)
device = "cuda" if torch.cuda.is_available() else "cpu"
cent_model.to(device)
ds = DatasetFromNumpy(X_pool_pp, pooled_y_train)
dl = DataLoader(ds, batch_size=64, shuffle=True)
opt = torch.optim.AdamW(cent_model.parameters(), lr=BEST_MLP_LR, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=90, eta_min=1e-6)
crit = nn.CrossEntropyLoss()
best_loss, best_sd, patience = float("inf"), None, 0
for ep in range(100):
    cent_model.train()
    for xb, yb in dl:
        xb, yb = xb.to(device), yb.to(device)
        if ep < 10:
            for pg in opt.param_groups: pg["lr"] = BEST_MLP_LR * (ep + 1) / 10
        opt.zero_grad(); crit(cent_model(xb), yb).backward(); opt.step()
    if ep >= 10: sched.step()
    if ep % 5 == 0:
        cent_model.eval()
        with torch.no_grad():
            vl = sum(crit(cent_model(xb.to(device)), yb.to(device)).item() for xb, yb in dl) / len(dl)
        if vl < best_loss:
            best_loss = vl; best_sd = {k: v.cpu().clone() for k, v in cent_model.state_dict().items()}; patience = 0
        else:
            patience += 1
            if patience >= 3: break
if best_sd: cent_model.load_state_dict(best_sd)
cent_model.eval()

def mlp_predict_proba(model, X_np, dev=device):
    model.eval()
    X_t = torch.tensor(X_np, dtype=torch.float32).to(dev)
    with torch.no_grad():
        return F.softmax(model(X_t), dim=1).cpu().numpy()[:, 1]

centralized_mlp_results = {}
for site in SITE_ORDER:
    X_tt, y_tt = pooled_test_sets[site]
    proba = mlp_predict_proba(cent_model, X_tt)
    centralized_mlp_results[f"{site}_BalAcc"] = balanced_accuracy_score(y_tt, proba >= BEST_MLP_THRESH)
    centralized_mlp_results[f"{site}_AUC"] = roc_auc_score(y_tt, proba)
proba_all = mlp_predict_proba(cent_model, combined_X_test)
centralized_mlp_results["All_BalAcc"] = balanced_accuracy_score(combined_y_test, proba_all >= BEST_MLP_THRESH)
centralized_mlp_results["All_AUC"] = roc_auc_score(combined_y_test, proba_all)
print("Centralized MLP results:")
for site in SITE_ORDER:
    print(f"  {site}: BalAcc={centralized_mlp_results[f'{site}_BalAcc']:.4f}  AUC={centralized_mlp_results[f'{site}_AUC']:.4f}")
print(f"  All: BalAcc={centralized_mlp_results['All_BalAcc']:.4f}  AUC={centralized_mlp_results['All_AUC']:.4f}")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# CENTRALIZED RF (pooled data, Option-A params)
# ═══════════════════════════════════════════════════════════════════════════

print("\n=== Centralized RF ===")
rf_cent = RandomForestClassifier(**RF_PARAMS, random_state=SEED, n_jobs=-1)
rf_cent.fit(X_pool_pp, pooled_y_train)

centralized_rf_results = {}
for site in SITE_ORDER:
    X_tt, y_tt = pooled_test_sets[site]
    proba = rf_cent.predict_proba(X_tt)[:, 1]
    centralized_rf_results[f"{site}_BalAcc"] = balanced_accuracy_score(y_tt, proba >= BEST_RF_THRESH)
    centralized_rf_results[f"{site}_AUC"] = roc_auc_score(y_tt, proba)
proba_all = rf_cent.predict_proba(combined_X_test)[:, 1]
centralized_rf_results["All_BalAcc"] = balanced_accuracy_score(combined_y_test, proba_all >= BEST_RF_THRESH)
centralized_rf_results["All_AUC"] = roc_auc_score(combined_y_test, proba_all)
print("Centralized RF results:")
for site in SITE_ORDER:
    print(f"  {site}: BalAcc={centralized_rf_results[f'{site}_BalAcc']:.4f}  AUC={centralized_rf_results[f'{site}_AUC']:.4f}")
print(f"  All: BalAcc={centralized_rf_results['All_BalAcc']:.4f}  AUC={centralized_rf_results['All_AUC']:.4f}")

In [ ]:
# ── Model construction + serialisation helpers ──
def build_mlp():
    return SpectralAttentionMLP(
        input_dim=6000, n_classes=2, hidden_dim=512, head_dims=(256, 128),
        dropout_high=BEST_MLP_DH, dropout_low=BEST_MLP_DH/2.0, use_attention=False)

def model_to_numpy(model):
    return [v.cpu().numpy() for v in model.state_dict().values()]

def numpy_to_model(model, params):
    sd = model.state_dict()
    for k, p in zip(sd.keys(), params):
        sd[k] = torch.tensor(p)
    model.load_state_dict(sd)

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# LOCAL MLP TRAINING (mixing + FedProx proximal term)
# ═══════════════════════════════════════════════════════════════════════════

def train_local_mixed(model, X_np, y_np, n_mixes, proximal_mu, global_params):
    """Train model on (X_np, y_np) for n_mixes epochs, each with different seed.
    If n_mixes > 1, average state_dicts. Returns final model state_dict + avg loss."""
    collect_sds = []
    total_loss = 0.0
    for mix_i in range(n_mixes):
        seed = SEED + mix_i * 100 + int(proximal_mu * 1000)
        torch.manual_seed(seed); np.random.seed(seed)
        # Shuffle indices
        idx = np.random.permutation(len(X_np))
        X_shuf, y_shuf = X_np[idx], y_np[idx]
        ds = DatasetFromNumpy(X_shuf, y_shuf)
        dl = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False)  # already shuffled

        opt = torch.optim.AdamW(model.parameters(), lr=BEST_MLP_LR, weight_decay=1e-4)
        crit = nn.CrossEntropyLoss()
        model.train()
        batch_loss = 0.0; n_batch = 0
        for xb, yb in dl:
            xb, yb = xb.to(FL_DEVICE), yb.to(FL_DEVICE)
            opt.zero_grad()
            loss = crit(model(xb), yb)
            if proximal_mu > 0 and global_params is not None:
                prox = sum((w - gw.to(FL_DEVICE)).norm(2)
                          for w, gw in zip(model.parameters(), global_params))
                loss = loss + (proximal_mu / 2.0) * prox
            loss.backward(); opt.step()
            batch_loss += loss.item(); n_batch += 1
        total_loss += batch_loss / n_batch
        collect_sds.append({k: v.cpu().clone() for k, v in model.state_dict().items()})

    # Average state dicts if multiple mixes
    if len(collect_sds) > 1:
        avg_sd = {}
        for k in collect_sds[0].keys():
            avg_sd[k] = torch.stack([sd[k] for sd in collect_sds]).mean(0)
        model.load_state_dict(avg_sd)
    return total_loss / n_mixes

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# FED MLP CLIENT
# ═══════════════════════════════════════════════════════════════════════════

class FedMLPClient(fl.client.NumPyClient):
    def __init__(self, cid, X_train, y_train):
        self.cid = cid
        self.X_train, self.y_train = X_train, y_train
        self.n_mixes = n_mixes(len(X_train))
        self.model = build_mlp().to(FL_DEVICE)

    def get_parameters(self, config):
        return model_to_numpy(self.model)

    def set_parameters(self, params):
        numpy_to_model(self.model, params)

    def fit(self, parameters, config):
        self.set_parameters(parameters)
        proximal_mu = float(config.get("proximal_mu", 0.0))
        global_copy = None
        if proximal_mu > 0:
            global_copy = [p.clone().detach() for p in self.model.parameters()]
        loss = train_local_mixed(self.model, self.X_train, self.y_train,
                                 self.n_mixes, proximal_mu, global_copy)
        return (self.get_parameters({}), len(self.X_train),
                {"train_loss": loss, "n_mixes": self.n_mixes})

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# FED LR CLIENT
# ═══════════════════════════════════════════════════════════════════════════

class FedLRClient(fl.client.NumPyClient):
    def __init__(self, cid, X_train, y_train):
        self.cid = cid; self.X_train, self.y_train = X_train, y_train
        nf = X_train.shape[1]
        self.model = LogisticRegression(C=BEST_LR_C, penalty="l2", solver="saga",
                                        max_iter=1, warm_start=True,
                                        class_weight="balanced", random_state=SEED)
        self.model.classes_ = np.array([0, 1])
        self.model.coef_ = np.zeros((1, nf)); self.model.intercept_ = np.zeros(1)

    def get_parameters(self, config):
        return [self.model.coef_.ravel(), self.model.intercept_]

    def set_parameters(self, params):
        self.model.coef_ = params[0].reshape(1, -1); self.model.intercept_ = params[1]

    def fit(self, parameters, config):
        self.set_parameters(parameters)
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            self.model.fit(self.X_train, self.y_train)
        return (self.get_parameters({}), len(self.X_train), {"num_examples": len(self.X_train)})

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# FED RF CLIENT + TREE COLLECTION STRATEGY
# ═══════════════════════════════════════════════════════════════════════════

def trees_to_array(estimators):
    buf = io.BytesIO(); joblib.dump(estimators, buf); buf.seek(0)
    return np.frombuffer(buf.read(), dtype=np.uint8)

def array_to_trees(arr):
    return joblib.load(io.BytesIO(arr.tobytes()))

class TreeCollectionFedAvg(fl.server.strategy.FedAvg):
    def aggregate_fit(self, server_round, results, failures):
        if not results: return None, {}
        all_trees = []
        for _, fit_res in results:
            ndarrays = fl.common.parameters_to_ndarrays(fit_res.parameters)
            if len(ndarrays[0]) > 0:
                all_trees.extend(array_to_trees(ndarrays[0]))
        combined = trees_to_array(all_trees)
        aggregated = fl.common.ndarrays_to_parameters([combined])
        metrics = {"total_trees": len(all_trees)}
        if self.fit_metrics_aggregation_fn:
            metrics.update(self.fit_metrics_aggregation_fn(
                [fit_res.metrics for _, fit_res in results]))
        return aggregated, metrics

class FedRFClient(fl.client.NumPyClient):
    def __init__(self, cid, X_train, y_train):
        self.cid = cid; self.X_train, self.y_train = X_train, y_train

    def get_parameters(self, config):
        return [np.array([], dtype=np.uint8)]

    def fit(self, parameters, config):
        rf = RandomForestClassifier(**RF_PARAMS, random_state=SEED + int(self.cid),
                                     n_jobs=-1, warm_start=True)
        rf.fit(self.X_train, self.y_train)
        new_trees = list(rf.estimators_)
        return ([trees_to_array(new_trees)], len(self.X_train),
                {"n_new_trees": len(new_trees), "num_examples": len(self.X_train)})

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# CHECKPOINT FEDAVG / FEDPROX  (saves per-client + global models)
# ═══════════════════════════════════════════════════════════════════════════

class CheckpointFedAvg(fl.server.strategy.FedAvg):
    def __init__(self, strategy_name, model_dir, **kwargs):
        super().__init__(**kwargs)
        self.strategy_name = strategy_name
        self.model_dir = Path(model_dir) / strategy_name
        self.model_dir.mkdir(parents=True, exist_ok=True)

    def aggregate_fit(self, server_round, results, failures):
        round_dir = self.model_dir / f"round_{server_round:03d}"
        round_dir.mkdir(parents=True, exist_ok=True)
        # Save per-client weights
        for cp, fit_res in results:
            cid = cp.cid
            ndarrays = fl.common.parameters_to_ndarrays(fit_res.parameters)
            m = build_mlp(); numpy_to_model(m, ndarrays)
            torch.save(m.state_dict(), round_dir / f"client_{cid}.pt")
        aggregated, metrics = super().aggregate_fit(server_round, results, failures)
        # Save global model
        if aggregated is not None:
            nd = fl.common.parameters_to_ndarrays(aggregated)
            gm = build_mlp(); numpy_to_model(gm, nd)
            torch.save(gm.state_dict(), round_dir / "global_model.pt")
        return aggregated, metrics

class CheckpointFedProx(fl.server.strategy.FedProx):
    def __init__(self, strategy_name, model_dir, **kwargs):
        super().__init__(**kwargs)
        self.strategy_name = strategy_name
        self.model_dir = Path(model_dir) / strategy_name
        self.model_dir.mkdir(parents=True, exist_ok=True)

    def aggregate_fit(self, server_round, results, failures):
        round_dir = self.model_dir / f"round_{server_round:03d}"
        round_dir.mkdir(parents=True, exist_ok=True)
        for cp, fit_res in results:
            cid = cp.cid
            ndarrays = fl.common.parameters_to_ndarrays(fit_res.parameters)
            m = build_mlp(); numpy_to_model(m, ndarrays)
            torch.save(m.state_dict(), round_dir / f"client_{cid}.pt")
        aggregated, metrics = super().aggregate_fit(server_round, results, failures)
        if aggregated is not None:
            nd = fl.common.parameters_to_ndarrays(aggregated)
            gm = build_mlp(); numpy_to_model(gm, nd)
            torch.save(gm.state_dict(), round_dir / "global_model.pt")
        return aggregated, metrics

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# CHECKPOINT LR  (saves per-client + global coefs)
# ═══════════════════════════════════════════════════════════════════════════

class CheckpointLRFedAvg(fl.server.strategy.FedAvg):
    def __init__(self, model_dir, **kwargs):
        super().__init__(**kwargs)
        self.model_dir = Path(model_dir) / "fedavg_lr"
        self.model_dir.mkdir(parents=True, exist_ok=True)

    def aggregate_fit(self, server_round, results, failures):
        round_dir = self.model_dir / f"round_{server_round:03d}"
        round_dir.mkdir(parents=True, exist_ok=True)
        for cp, fit_res in results:
            ndarrays = fl.common.parameters_to_ndarrays(fit_res.parameters)
            np.savez(round_dir / f"client_{cp.cid}.npz", coef=ndarrays[0], intercept=ndarrays[1])
        aggregated, metrics = super().aggregate_fit(server_round, results, failures)
        if aggregated is not None:
            nd = fl.common.parameters_to_ndarrays(aggregated)
            np.savez(round_dir / "global_model.npz", coef=nd[0], intercept=nd[1])
        return aggregated, metrics

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# CHECKPOINT RF TREE COLLECTION
# ═══════════════════════════════════════════════════════════════════════════

class CheckpointTreeCollection(TreeCollectionFedAvg):
    def __init__(self, model_dir, **kwargs):
        super().__init__(**kwargs)
        self.model_dir = Path(model_dir) / "fedrf"
        self.model_dir.mkdir(parents=True, exist_ok=True)

    def aggregate_fit(self, server_round, results, failures):
        round_dir = self.model_dir / f"round_{server_round:03d}"
        round_dir.mkdir(parents=True, exist_ok=True)
        for cp, fit_res in results:
            ndarrays = fl.common.parameters_to_ndarrays(fit_res.parameters)
            if len(ndarrays[0]) > 0:
                np.save(round_dir / f"client_{cp.cid}_trees.npy", ndarrays[0])
        aggregated, metrics = super().aggregate_fit(server_round, results, failures)
        if aggregated is not None:
            nd = fl.common.parameters_to_ndarrays(aggregated)
            np.save(round_dir / "global_trees.npy", nd[0])
        # Save metrics
        return aggregated, metrics

In [ ]:
# ── Server-side evaluation functions ──

def get_mlp_eval_fn(test_dict, threshold, hist_list):
    def evaluate(server_round, parameters, config):
        m = build_mlp(); numpy_to_model(m, parameters); m.to(FL_DEVICE); m.eval()
        record = {"round": server_round}
        ap, al = [], []
        for site in SITE_ORDER:
            X_tt, y_tt = test_dict[site]
            proba = mlp_predict_proba(m, X_tt, dev=FL_DEVICE)
            preds = proba >= threshold
            record[f"{site}_BalAcc"] = float(balanced_accuracy_score(y_tt, preds))
            record[f"{site}_AUC"] = float(roc_auc_score(y_tt, proba))
            ap.append(proba); al.append(y_tt)
        apc = np.concatenate(ap); alc = np.concatenate(al)
        record["All_BalAcc"] = float(balanced_accuracy_score(alc, apc >= threshold))
        record["All_AUC"] = float(roc_auc_score(alc, apc))
        # Save metrics.json
        rd = Path(MODEL_DIR) / '__current_strategy__' / f"round_{server_round:03d}"
        if rd.exists():
            import json as _json
            with open(rd / "metrics.json", "w") as f:
                _json.dump(record, f)
        hist_list.append(record)
        return (1.0 - record["All_BalAcc"], record)
    return evaluate

def get_lr_eval_fn(test_dict, threshold, hist_list):
    def evaluate(server_round, parameters, config):
        lr = LogisticRegression(C=BEST_LR_C, penalty="l2", solver="lbfgs",
                                class_weight="balanced", max_iter=5000)
        lr.classes_ = np.array([0, 1])
        lr.coef_ = parameters[0].reshape(1, -1); lr.intercept_ = parameters[1]
        record = {"round": server_round}
        ap, al = [], []
        for site in SITE_ORDER:
            X_tt, y_tt = test_dict[site]
            proba = lr.predict_proba(X_tt)[:, 1]
            preds = proba >= threshold
            record[f"{site}_BalAcc"] = float(balanced_accuracy_score(y_tt, preds))
            record[f"{site}_AUC"] = float(roc_auc_score(y_tt, proba))
            ap.append(proba); al.append(y_tt)
        apc = np.concatenate(ap); alc = np.concatenate(al)
        record["All_BalAcc"] = float(balanced_accuracy_score(alc, apc >= threshold))
        record["All_AUC"] = float(roc_auc_score(alc, apc))
        hist_list.append(record)
        return (1.0 - record["All_BalAcc"], record)
    return evaluate

def get_rf_eval_fn(test_dict, threshold, hist_list):
    def evaluate(server_round, parameters, config):
        if len(parameters[0]) == 0:
            return (1.0, {"All_BalAcc": 0.5})
        trees = array_to_trees(parameters[0])
        rf = RandomForestClassifier(**RF_PARAMS, n_jobs=-1)
        rf.estimators_ = trees; rf.n_classes_ = 2
        rf.classes_ = np.array([0, 1]); rf.n_outputs_ = 1
        record = {"round": server_round, "n_trees": len(trees)}
        ap, al = [], []
        for site in SITE_ORDER:
            X_tt, y_tt = test_dict[site]
            proba = rf.predict_proba(X_tt)[:, 1]
            preds = proba >= threshold
            record[f"{site}_BalAcc"] = float(balanced_accuracy_score(y_tt, preds))
            record[f"{site}_AUC"] = float(roc_auc_score(y_tt, proba))
            ap.append(proba); al.append(y_tt)
        apc = np.concatenate(ap); alc = np.concatenate(al)
        record["All_BalAcc"] = float(balanced_accuracy_score(alc, apc >= threshold))
        record["All_AUC"] = float(roc_auc_score(alc, apc))
        hist_list.append(record)
        return (1.0 - record["All_BalAcc"], record)
    return evaluate

In [ ]:
# ── Client factories ──
def mlp_client_fn(cid):
    site = SITE_ORDER[int(cid)]
    X_tr, y_tr = client_train_pp[site]
    return FedMLPClient(cid, X_tr, y_tr).to_client()

def lr_client_fn(cid):
    site = SITE_ORDER[int(cid)]
    X_tr, y_tr = client_train_pp[site]
    return FedLRClient(cid, X_tr, y_tr).to_client()

def rf_client_fn(cid):
    site = SITE_ORDER[int(cid)]
    X_tr, y_tr = client_train_pp[site]
    return FedRFClient(cid, X_tr, y_tr).to_client()

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# RUN: FedAvg MLP
# ═══════════════════════════════════════════════════════════════════════════

print("\n=== FedAvg MLP ===")
eval_hist_fedavg = []
_eval_fn = get_mlp_eval_fn(client_test_pp, BEST_MLP_THRESH, eval_hist_fedavg)

strategy = CheckpointFedAvg("fedavg_mlp", MODEL_DIR,
    fraction_fit=1.0, fraction_evaluate=1.0,
    min_fit_clients=4, min_evaluate_clients=4, min_available_clients=4,
    evaluate_fn=_eval_fn,
    initial_parameters=fl.common.ndarrays_to_parameters(model_to_numpy(build_mlp())))

strategy.model_dir = Path(MODEL_DIR) / "fedavg_mlp"  # re-set after init

fl.simulation.start_simulation(
    client_fn=mlp_client_fn, num_clients=4,
    config=fl.server.ServerConfig(num_rounds=NUM_ROUNDS),
    strategy=strategy, client_resources={"num_cpus": 1, "num_gpus": 0})

fedavg_hist = eval_hist_fedavg.copy()
print(f"FedAvg MLP done. {len(fedavg_hist)} rounds.")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# RUN: FedProx MLP (mu=0.01, 0.1, 0.5)
# ═══════════════════════════════════════════════════════════════════════════

fedprox_histories = {}
for mu in FEDPROX_MUS:
    print(f"\n=== FedProx MLP (mu={mu}) ===")
    eval_hist_fedprox = []
    _eval_fn = get_mlp_eval_fn(client_test_pp, BEST_MLP_THRESH, eval_hist_fedprox)

    strategy = CheckpointFedProx(f"fedprox_mlp_mu{mu}", MODEL_DIR,
        fraction_fit=1.0, fraction_evaluate=1.0,
        min_fit_clients=4, min_evaluate_clients=4, min_available_clients=4,
        proximal_mu=mu, evaluate_fn=_eval_fn,
        initial_parameters=fl.common.ndarrays_to_parameters(model_to_numpy(build_mlp())))

    strategy.model_dir = Path(MODEL_DIR) / f"fedprox_mlp_mu{mu}"
    fl.simulation.start_simulation(
        client_fn=mlp_client_fn, num_clients=4,
        config=fl.server.ServerConfig(num_rounds=NUM_ROUNDS),
        strategy=strategy, client_resources={"num_cpus": 1, "num_gpus": 0})
    fedprox_histories[mu] = eval_hist_fedprox.copy()
    print(f"FedProx mu={mu} done. {len(eval_hist_fedprox)} rounds.")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# RUN: FedAvg LR
# ═══════════════════════════════════════════════════════════════════════════

print("\n=== FedAvg LR ===")
eval_hist_fedlr = []
_eval_fn = get_lr_eval_fn(client_test_pp, BEST_LR_THRESH, eval_hist_fedlr)

nr_feat = client_train_pp[SITE_ORDER[0]][0].shape[1]
lr_init = LogisticRegression(C=BEST_LR_C, penalty="l2", solver="saga",
                              max_iter=1, warm_start=True, class_weight="balanced", random_state=SEED)
lr_init.classes_ = np.array([0, 1]); lr_init.coef_ = np.zeros((1, nr_feat)); lr_init.intercept_ = np.zeros(1)
lr_init_params = [lr_init.coef_.ravel(), lr_init.intercept_]

strategy = CheckpointLRFedAvg(MODEL_DIR,
    fraction_fit=1.0, fraction_evaluate=1.0,
    min_fit_clients=4, min_evaluate_clients=4, min_available_clients=4,
    evaluate_fn=_eval_fn,
    initial_parameters=fl.common.ndarrays_to_parameters(lr_init_params))

strategy.model_dir = Path(MODEL_DIR) / "fedavg_lr"
fl.simulation.start_simulation(
    client_fn=lr_client_fn, num_clients=4,
    config=fl.server.ServerConfig(num_rounds=NUM_ROUNDS),
    strategy=strategy, client_resources={"num_cpus": 1, "num_gpus": 0})

fedlr_hist = eval_hist_fedlr.copy()
print(f"FedAvg LR done. {len(fedlr_hist)} rounds.")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# RUN: FedRF (tree collection)
# ═══════════════════════════════════════════════════════════════════════════

print("\n=== FedRF (Tree Collection) ===")
eval_hist_fedrf = []
_eval_fn = get_rf_eval_fn(client_test_pp, BEST_RF_THRESH, eval_hist_fedrf)

strategy = CheckpointTreeCollection(MODEL_DIR,
    fraction_fit=1.0, fraction_evaluate=1.0,
    min_fit_clients=4, min_evaluate_clients=4, min_available_clients=4,
    evaluate_fn=_eval_fn,
    initial_parameters=fl.common.ndarrays_to_parameters([np.array([], dtype=np.uint8)]))

strategy.model_dir = Path(MODEL_DIR) / "fedrf"
fl.simulation.start_simulation(
    client_fn=rf_client_fn, num_clients=4,
    config=fl.server.ServerConfig(num_rounds=NUM_RF_ROUNDS),
    strategy=strategy, client_resources={"num_cpus": 1, "num_gpus": 0})

fedrf_hist = eval_hist_fedrf.copy()
print(f"FedRF done. {len(fedrf_hist)} rounds.")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# CROSS-SITE: Train MLP on A only, test B/C/D
# ═══════════════════════════════════════════════════════════════════════════

print("\n=== Cross-Site MLP (A → B/C/D) ===")
# Preprocessing fit on A train only
X_A_train, y_A_train = client_train["A"]
state_cs = fit_input_transform(X_A_train, "log1p+standardize")
X_A_tr = apply_input_transform(X_A_train, state_cs)

cross_test_sets = {}
for site in "BCD":
    X_te, y_te = client_test[site]
    cross_test_sets[site] = (apply_input_transform(X_te, state_cs), y_te)
X_a_val, y_a_val = client_test["A"]
cross_test_sets["A-val"] = (apply_input_transform(X_a_val, state_cs), y_a_val)

cs_mlp = build_mlp()
dev_cs = "cuda" if torch.cuda.is_available() else "cpu"; cs_mlp.to(dev_cs)
ds_cs = DatasetFromNumpy(X_A_tr, y_A_train)
dl_cs = DataLoader(ds_cs, batch_size=64, shuffle=True)
opt_cs = torch.optim.AdamW(cs_mlp.parameters(), lr=BEST_MLP_LR, weight_decay=1e-4)
sched_cs = torch.optim.lr_scheduler.CosineAnnealingLR(opt_cs, T_max=90, eta_min=1e-6)
crit_cs = nn.CrossEntropyLoss()
for ep in range(100):
    cs_mlp.train()
    for xb, yb in dl_cs:
        xb, yb = xb.to(dev_cs), yb.to(dev_cs)
        if ep < 10:
            for pg in opt_cs.param_groups: pg["lr"] = BEST_MLP_LR * (ep + 1) / 10
        opt_cs.zero_grad(); crit_cs(cs_mlp(xb), yb).backward(); opt_cs.step()
    if ep >= 10: sched_cs.step()
cs_mlp.eval()

cross_site_results = {}
for name, (X_tt, y_tt) in cross_test_sets.items():
    proba = mlp_predict_proba(cs_mlp, X_tt, dev=dev_cs)
    cross_site_results[f"{name}_BalAcc"] = balanced_accuracy_score(y_tt, proba >= BEST_MLP_THRESH)
    cross_site_results[f"{name}_AUC"] = roc_auc_score(y_tt, proba)
    print(f"  {name}: BalAcc={cross_site_results[f'{name}_BalAcc']:.4f}  AUC={cross_site_results[f'{name}_AUC']:.4f}")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# CROSS-SITE: Train RF on A only, test B/C/D
# ═══════════════════════════════════════════════════════════════════════════

print("\n=== Cross-Site RF (A → B/C/D) ===")
rf_cs = RandomForestClassifier(**RF_PARAMS, random_state=SEED, n_jobs=-1)
rf_cs.fit(X_A_tr, y_A_train)

cross_site_rf_results = {}
for name, (X_tt, y_tt) in cross_test_sets.items():
    proba = rf_cs.predict_proba(X_tt)[:, 1]
    cross_site_rf_results[f"{name}_BalAcc"] = balanced_accuracy_score(y_tt, proba >= BEST_RF_THRESH)
    cross_site_rf_results[f"{name}_AUC"] = roc_auc_score(y_tt, proba)
    print(f"  {name}: BalAcc={cross_site_rf_results[f'{name}_BalAcc']:.4f}  AUC={cross_site_rf_results[f'{name}_AUC']:.4f}")

In [ ]:
# ── Assemble final results DataFrame ──
def last_metrics(h):
    return h[-1] if h else {}

rows = []

def add_row(method, cs_source=None, fed_source=None):
    r = {"Method": method}
    for site in SITE_ORDER:
        if cs_source:
            r[f"{site}_BalAcc"] = cs_source.get(f"{site}_BalAcc", cs_source.get(f"Site-{site}_BalAcc", np.nan))
            r[f"{site}_AUC"] = cs_source.get(f"{site}_AUC", cs_source.get(f"Site-{site}_AUC", np.nan))
        elif fed_source:
            lm = last_metrics(fed_source)
            r[f"{site}_BalAcc"] = lm.get(f"{site}_BalAcc", np.nan)
            r[f"{site}_AUC"] = lm.get(f"{site}_AUC", np.nan)
    if fed_source:
        lm = last_metrics(fed_source)
        r["All_BalAcc"] = lm.get("All_BalAcc", np.nan)
        r["All_AUC"] = lm.get("All_AUC", np.nan)
    elif cs_source:
        r["All_BalAcc"] = np.nan; r["All_AUC"] = np.nan
    rows.append(r)

add_row("Centralized MLP", cs_source=centralized_mlp_results)
add_row("Centralized RF", cs_source=centralized_rf_results)
add_row("FL FedAvg (MLP)", fed_source=fedavg_hist)
add_row("FL FedProx mu=0.1 (MLP)", fed_source=fedprox_histories[0.1])
add_row("FL FedProx mu=0.5 (MLP)", fed_source=fedprox_histories[0.5])
add_row("FL FedAvg (LR)", fed_source=fedlr_hist)
add_row("FL FedRF (Trees)", fed_source=fedrf_hist)
add_row("Cross-Site MLP", cs_source=cross_site_results)
add_row("Cross-Site RF", cs_source=cross_site_rf_results)

df_results = pd.DataFrame(rows)
cols = ["Method"] + [f"{s}_BalAcc" for s in SITE_ORDER] + ["All_BalAcc"]
print(df_results[cols].to_string(index=False))

In [ ]:
# ── Convergence plot ──
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Panel 1: FedAvg per-site
ax = axes[0]
for site in SITE_ORDER:
    vals = [h.get(f"{site}_BalAcc", np.nan) for h in fedavg_hist]
    ax.plot(range(1, len(vals)+1), vals, marker='.', label=f"Site {site}")
vals_all = [h.get("All_BalAcc", np.nan) for h in fedavg_hist]
ax.plot(range(1, len(vals_all)+1), vals_all, 'k-', lw=2, label="All")
ax.set_title("FedAvg MLP — Per-Site Convergence"); ax.set_xlabel("Round"); ax.set_ylabel("BalAcc")
ax.legend(fontsize=7); ax.grid(True, ls='--', alpha=0.5); ax.set_ylim(0.3, 1.0)

# Panel 2: All FL methods, All BalAcc
ax = axes[1]
for label, hist, c, ls in [
    ("FedAvg MLP", fedavg_hist, "#ff7f0e", "-"),
    ("FedProx mu=0.1", fedprox_histories[0.1], "#d62728", "--"),
    ("FedAvg LR", fedlr_hist, "#1f77b4", "-."),
    ("FedRF", fedrf_hist, "#2ca02c", "-"),
]:
    vals = [h.get("All_BalAcc", np.nan) for h in hist]
    ax.plot(range(1, len(vals)+1), vals, color=c, ls=ls, lw=2, label=label)
ax.axhline(centralized_mlp_results["All_BalAcc"], color='gray', ls=':', lw=2,
           label=f'Centralized MLP')
ax.axhline(centralized_rf_results["All_BalAcc"], color='gray', ls='--', lw=2,
           label=f'Centralized RF')
ax.set_title("All Methods — All-Site BalAcc"); ax.set_xlabel("Round"); ax.set_ylabel("BalAcc")
ax.legend(fontsize=7); ax.grid(True, ls='--', alpha=0.5); ax.set_ylim(0.3, 1.0)

fig.suptitle(f"{DRUG_NAME} — Federated Learning Convergence", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(OUT_DIR / "convergence.pdf", bbox_inches="tight")
plt.show()

In [ ]:
# ── Heatmap: Balanced Accuracy ──
ba_data = {}
for _, r in df_results.iterrows():
    ba_data[r["Method"]] = {f"Site {s}": r[f"{s}_BalAcc"] for s in SITE_ORDER}
    if not np.isnan(r.get("All_BalAcc", np.nan)):
        ba_data[r["Method"]]["All"] = r["All_BalAcc"]

df_ba_hm = pd.DataFrame(ba_data).T
df_ba_hm = df_ba_hm[[c for c in [f"Site {s}" for s in SITE_ORDER] + ["All"] if c in df_ba_hm.columns]]

fig, ax = plt.subplots(figsize=(10, max(4, len(df_ba_hm)*0.5)))
sns.heatmap(df_ba_hm, annot=True, fmt=".3f", cmap="RdYlGn", vmin=0.4, vmax=1.0,
            linewidths=1.0, linecolor="white",
            cbar_kws={"label": "Balanced Accuracy"}, ax=ax)
ax.set_title(f"{DRUG_NAME} — Balanced Accuracy", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(OUT_DIR / "heatmap_balacc.pdf", bbox_inches="tight")
plt.show()

In [ ]:
# ── Heatmap: AUC ──
auc_data = {}
for _, r in df_results.iterrows():
    auc_data[r["Method"]] = {f"Site {s}": r[f"{s}_AUC"] for s in SITE_ORDER}
    if not np.isnan(r.get("All_AUC", np.nan)):
        auc_data[r["Method"]]["All"] = r["All_AUC"]

df_auc_hm = pd.DataFrame(auc_data).T
df_auc_hm = df_auc_hm[[c for c in [f"Site {s}" for s in SITE_ORDER] + ["All"] if c in df_auc_hm.columns]]

fig, ax = plt.subplots(figsize=(10, max(4, len(df_auc_hm)*0.5)))
sns.heatmap(df_auc_hm, annot=True, fmt=".3f", cmap="RdYlGn", vmin=0.4, vmax=1.0,
            linewidths=1.0, linecolor="white",
            cbar_kws={"label": "AUC-ROC"}, ax=ax)
ax.set_title(f"{DRUG_NAME} — AUC-ROC", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(OUT_DIR / "heatmap_auc.pdf", bbox_inches="tight")
plt.show()

In [ ]:
# ── Save results ──
df_results.to_csv(OUT_DIR / "final_results.csv", index=False)
pd.DataFrame(fedavg_hist).to_csv(OUT_DIR / "fedavg_per_round.csv", index=False)
pd.DataFrame(fedlr_hist).to_csv(OUT_DIR / "fedlr_per_round.csv", index=False)
pd.DataFrame(fedrf_hist).to_csv(OUT_DIR / "fedrf_per_round.csv", index=False)
for mu in FEDPROX_MUS:
    pd.DataFrame(fedprox_histories[mu]).to_csv(OUT_DIR / f"fedprox_mu{mu}_per_round.csv", index=False)

# Save best params used
with open(OUT_DIR / "best_params_used.txt", "w") as f:
    f.write(f"DRUG={DRUG_NAME}\n")
    f.write(f"BEST_LR_C={BEST_LR_C}\n")
    f.write(f"BEST_MLP_LR={BEST_MLP_LR}\n")
    f.write(f"BEST_MLP_DH={BEST_MLP_DH}\n")
    f.write(f"BEST_LR_THRESH={BEST_LR_THRESH}\n")
    f.write(f"BEST_MLP_THRESH={BEST_MLP_THRESH}\n")
    f.write(f"RF_PARAMS={RF_PARAMS}\n")
    f.write(f"BEST_RF_THRESH={BEST_RF_THRESH}\n")

print("\n" + "="*60)
print(f"  Done. Results in {OUT_DIR.resolve()}")
print(f"  Models in {MODEL_DIR.resolve()}")
for f in sorted(OUT_DIR.glob("*")):
    print(f"    {f.name}")
print("\nModel directories:")
for d in sorted(MODEL_DIR.glob("*")):
    ndirs = len(list(d.glob("round_*")))
    print(f"    {d.name}/ ({ndirs} rounds)")

---
**Done.** Federated analysis for **Gentamicin** complete.

| Output | Location |
|---|---|
| Results CSV | `results/` |
| Convergence plot | `results/convergence.pdf` |
| Heatmaps | `results/heatmap_balacc.pdf`, `results/heatmap_auc.pdf` |
| Model checkpoints | `models/{strategy}/round_NNN/` |
| Best params | `results/best_params_used.txt` |